<a href="https://colab.research.google.com/github/1KVueltasAlCampo/1KvueltasAlCampo.github.io/blob/main/Sistema_Recomendador_Preventivo_Cl%C3%ADnico_Deportivo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏋️ Sistema Recomendador Preventivo Clínico-Deportivo

## 1. Fuente de Datos y Contexto Clínico

Este cuaderno implementa un motor de recomendación de ejercicio basado en perfiles fisiológicos reales de reclutas del Ejército Brasileño. Los datos provienen de un estudio de cohorte prospectivo publicado en Mendeley Data ([Acceder al dataset](https://data.mendeley.com/datasets/26ty434tm5)).

### Variables del dataset original (140 reclutas)

| Variable | Tipo | Descripción |
|---|---|---|
| `mass` | Continua | Masa corporal (kg) |
| `BMC` | Continua | Contenido mineral óseo |
| `Body_fat_percentual` | Continua | Porcentaje de grasa corporal |
| `12-min_running` | Continua | Prueba de Cooper — capacidad cardiorrespiratoria (m) |
| `flexibility` | Continua | Flexibilidad (cm) |
| `1RM_strength` | Continua | Fuerza máxima — una repetición máxima (kg) |
| `plank_test` | Continua | Resistencia muscular isométrica (seg) |
| `Vertical_jump` | Continua | Potencia de tren inferior (cm) |
| `Balance` | Binaria | Déficit de balance funcional (`yes` / `no`) |
| `Previous_Physical_Activity_Level` | Ordinal | Nivel previo de actividad física |
| `History_Previous_Injury` | Binaria | Historial de lesión previa (`yes` / `no`) |
| `MSI_General` | Binaria | Lesión musculoesquelética general durante el seguimiento |
| `MSI_Trauma` |

In [11]:
import pandas as pd
import numpy as np
import requests
import io

# 1. Descargar el dataset desde Mendeley
url = "https://data.mendeley.com/public-files/datasets/26ty434tm5/files/ff117ca8-396f-444a-a5dc-6f2521c9df7a/file_downloaded"

print("Descargando dataset desde Mendeley...")
response = requests.get(url)
response.raise_for_status()

# Limpiar tabulaciones internas dentro de valores numéricos (67\t7 → 67,7)
contenido = response.content.decode('utf-8')
contenido_limpio = []
for linea in contenido.splitlines():
    partes = linea.split(';')
    partes_limpias = [p.replace('\t', ',') for p in partes]
    contenido_limpio.append(';'.join(partes_limpias))
contenido_final = '\n'.join(contenido_limpio)

# Cargar CSV con decimal=',' (formato europeo/brasileño)
df_real = pd.read_csv(io.StringIO(contenido_final), sep=';', decimal=',')
print(f"Dataset cargado: {len(df_real)} registros, {len(df_real.columns)} columnas\n")

# ── Definición correcta de variables ──────────────────────────────────────────
# Continuas: reciben ruido gaussiano
columnas_numericas = ['mass', 'BMC', 'Body_fat_percentual', '12-min_running',
                      'flexibility', '1RM_strength', 'plank_test', 'Vertical_jump']

# Categóricas binarias/ordinales: se copian sin modificación
# Balance = yes/no (presencia de déficit de balance funcional)
# History_Previous_Injury, MSI_General, MSI_Trauma, MSI_Overuse = yes/no
# Previous_Physical_Activity_Level = nivel ordinal
columnas_categoricas = ['Balance', 'Previous_Physical_Activity_Level',
                        'History_Previous_Injury', 'MSI_General', 'MSI_Trauma', 'MSI_Overuse']

print("Tipos de datos:")
print(df_real[columnas_numericas].dtypes)
print(f"\nDistribución de Balance:\n{df_real['Balance'].value_counts()}\n")

# 2. Generar datos sintéticos (10 copias con ruido gaussiano del 10%)
num_copias = 10
datos_sinteticos = []

for _ in range(num_copias):
    df_clon = df_real.copy()

    for col in columnas_numericas:
        desviacion = df_real[col].std() * 0.1
        ruido = np.random.normal(0, desviacion, size=len(df_clon))
        df_clon[col] = (df_clon[col] + ruido).round(2)
        # Evitar valores fisiológicamente imposibles
        df_clon[col] = df_clon[col].apply(lambda x: max(x, 1.0))

    # Las columnas categóricas se mantienen idénticas al original
    datos_sinteticos.append(df_clon)

# 3. Concatenar dataset final
df_final = pd.concat([df_real] + datos_sinteticos, ignore_index=True)

print(f"Dataset ampliado a {len(df_final)} registros.")
print(f"\nEstadísticas descriptivas (variables continuas):")
print(df_final[columnas_numericas].describe().round(2))
print(f"\nDistribución de Balance en dataset final:\n{df_final['Balance'].value_counts()}")

Descargando dataset desde Mendeley...
Dataset cargado: 140 registros, 15 columnas

Tipos de datos:
mass                   float64
BMC                    float64
Body_fat_percentual    float64
12-min_running           int64
flexibility            float64
1RM_strength             int64
plank_test               int64
Vertical_jump          float64
dtype: object

Distribución de Balance:
Balance
no     104
yes     36
Name: count, dtype: int64

Dataset ampliado a 1540 registros.

Estadísticas descriptivas (variables continuas):
          mass      BMC  Body_fat_percentual  12-min_running  flexibility  \
count  1540.00  1540.00              1540.00         1540.00      1540.00   
mean     66.64    21.85                12.62         2582.54        29.35   
std       9.88     3.06                 5.84          277.41         7.67   
min      46.21    16.51                 1.53         1890.88         1.00   
25%      59.96    19.74                 8.40         2423.55        25.33   
50%      

In [12]:
df_final.head()

,Subjects,mass,BMC,Body_fat_percentual,12-min_running,flexibility,1RM_strength,plank_test,Vertical_jump,Balance,Previous_Physical_Activity_Level,History_Previous_Injury,MSI_General,MSI_Trauma,MSI_Overuse
0,S1,67.7,22.4,11.6,3199.0,32.5,123.0,226.0,37.0,no,51,no,no,no,no
1,S2,63.4,21.9,16.0,2526.0,22.0,124.0,190.0,28.5,no,54,no,no,no,no
2,S3,59.9,20.2,6.7,2939.0,31.5,116.0,205.0,36.0,no,55,yes,no,no,no
3,S4,77.6,22.9,10.2,3186.0,1.5,177.0,140.0,34.3,yes,35,no,no,no,no
4,S5,66.6,22.5,9.5,2808.0,27.5,120.0,231.0,33.7,no,42,yes,yes,yes,no


## 2. Expansión del Dataset y Variable Puente de Equipamiento

### 2.1 Data Augmentation mediante Ruido Gaussiano Controlado

El dataset original cuenta con 140 registros, insuficientes para entrenar un motor de recomendación robusto. Para expandirlo a ~1,540 registros manteniendo la **fidelidad estadística**, se aplica la siguiente técnica de *Data Augmentation*:

Por cada variable continua, se generan nuevos registros sumando ruido extraído de una distribución normal cuya desviación estándar es el 10% de la desviación estándar original:

$$X_{\text{sintético}} = X_{\text{real}} + \mathcal{N}(0,\ (0.1 \cdot \sigma_{\text{real}})^2)$$

- Las **variables categóricas** (`Balance`, `MSI_*`, `History_Previous_Injury`) se copian sin modificación, preservando las proporciones clínicas originales.
- Se aplica un límite inferior de `1.0` para evitar valores fisiológicamente imposibles.

> Se generan **10 copias** del dataset original, resultando en 1,540 registros estadísticamente coherentes.

---

### 2.2 Variable Puente: `Equipment_Restriction`

Para conectar los perfiles fisiológicos del dataset clínico con el catálogo de ejercicios (`megaGymDataset`), se crea la columna `Equipment_Restriction` mediante reglas de kinesiología aplicadas a cada recluta:

| Condición de riesgo | Restricción asignada | Equipamiento permitido |
|---|---|---|
| `MSI_Overuse == yes` ó `Balance == yes` | `Restricted_Weight` | Bandas, cables, máquinas guiadas |
| Sin factores de riesgo | `Free_Weight_OK` | Barras, mancuernas, peso libre |

Esta columna actúa como **enlace directo** con la columna `Equipment` del catálogo de ejercicios: el algoritmo KNN agrupará usuarios por similitud fisiológica y el motor filtrará el catálogo según la restricción asignada, descartando automáticamente ejercicios incompatibles con el perfil de riesgo del usuario.

In [13]:
def asignar_restriccion(row):
    # Balance = 'yes' significa que el recluta TIENE déficit de balance funcional
    balance_pobre = str(row['Balance']).strip().lower() == 'yes'
    riesgo_sobreuso = str(row['MSI_Overuse']).strip().lower() == 'yes'

    if riesgo_sobreuso or balance_pobre:
        # Déficit de balance o historial de sobreuso → restringir peso libre
        return 'Restricted_Weight'
    else:
        return 'Free_Weight_OK'

df_final['Equipment_Restriction'] = df_final.apply(asignar_restriccion, axis=1)

# Verificar distribución resultante
print(df_final['Equipment_Restriction'].value_counts())
print(f"\nPorcentaje restringido: {(df_final['Equipment_Restriction'] == 'Restricted_Weight').mean()*100:.1f}%")

Equipment_Restriction
Free_Weight_OK       1034
Restricted_Weight     506
Name: count, dtype: int64

Porcentaje restringido: 32.9%


## 3. Normalización y Construcción del Espacio Vectorial de Usuarios

### 3.1 Separación de Variables: Inputs vs. Outputs

Antes de construir el espacio vectorial, es crítico separar las **variables de entrada** (features) de las **variables objetivo** (targets) para evitar *Data Leakage*.

Un atleta nuevo que llega al gimnasio **no tiene valores** para `MSI_Overuse`, `MSI_Trauma` o `Equipment_Restriction` — esas son etiquetas que el sistema debe *inferir*, no datos de entrada. Incluirlas en el cálculo de distancia KNN contaminaría el modelo con información que no existiría en producción.

| Rol | Columnas |
|---|---|
| **Inputs** (métricas de entrada) | `mass`, `BMC`, `Body_fat_percentual`, `12-min_running`, `flexibility`, `1RM_strength`, `plank_test`, `Vertical_jump`, `Balance`, `Previous_Physical_Activity_Level`, `History_Previous_Injury` |
| **Targets** (variables a predecir) | `MSI_General`, `MSI_Trauma`, `MSI_Overuse`, `Equipment_Restriction` |

---

### 3.2 Codificación de Variables Categóricas

Las variables binarias de texto (`yes`/`no`) se convierten a valores numéricos `1`/`0` para que sean compatibles con operaciones matemáticas:

- `Balance`: `yes → 1` (presenta déficit funcional), `no → 0`
- `History_Previous_Injury`: Variable binaria yes/no mapeada a 1/0 mediante codificación explícita
- `Previous_Physical_Activity_Level`: codificación binaria `yes/no → 1/0`

---

### 3.3 Normalización con StandardScaler

Para que la **distancia Euclidiana** de KNN funcione correctamente, todas las variables continuas deben estar en la misma escala. Sin normalización, variables con magnitudes grandes como `12-min_running` (aprox. 2,500 m) dominarían completamente sobre variables como `Body_fat_percentual` (~12%), haciendo invisibles sus contribuciones al cálculo de similitud.

Se utiliza **StandardScaler** (estandarización Z-score) porque los datos sintéticos fueron generados bajo una distribución gaussiana:

$$z = \frac{x - \mu}{\sigma}$$

Las variables booleanas codificadas (`Balance`, `History_Previous_Injury`) **no se escalan**, ya que su rango `{0, 1}` es intencional y consistente.

> El objeto `scaler` debe preservarse para normalizar de forma idéntica a cualquier atleta nuevo que ingrese al sistema en fase de inferencia.

In [14]:
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.preprocessing import StandardScaler
import pandas as pd

# -------------------------------------------------------------------
# 1. PREPARACIÓN DEL ESPACIO VECTORIAL
# -------------------------------------------------------------------
features_entrada = [
    'mass', 'BMC', 'Body_fat_percentual', '12-min_running', 'flexibility',
    '1RM_strength', 'plank_test', 'Vertical_jump', 'Balance',
    'Previous_Physical_Activity_Level', 'History_Previous_Injury'
]

targets = ['MSI_General', 'MSI_Trauma', 'MSI_Overuse', 'Equipment_Restriction']

df_inputs = df_final[features_entrada].copy()
df_targets = df_final[targets].copy()

# Binarias: Balance e History_Previous_Injury → yes=1, no=0
for col in ['Balance', 'History_Previous_Injury']:
    df_inputs[col] = df_inputs[col].apply(lambda x: 1 if str(x).strip().lower() == 'yes' else 0)

# Continuas a escalar (ahora incluye Previous_Physical_Activity_Level)
cols_a_escalar = [
    'mass', 'BMC', 'Body_fat_percentual', '12-min_running', 'flexibility',
    '1RM_strength', 'plank_test', 'Vertical_jump', 'Previous_Physical_Activity_Level'
]

scaler = StandardScaler()
df_inputs[cols_a_escalar] = scaler.fit_transform(df_inputs[cols_a_escalar])

print("Espacio Vectorial de Usuarios normalizado y listo para KNN:")
print(df_inputs.head().round(3))

Espacio Vectorial de Usuarios normalizado y listo para KNN:
    mass    BMC  Body_fat_percentual  12-min_running  flexibility  \
0  0.107  0.179               -0.174           2.223        0.411   
1 -0.328  0.015                0.579          -0.204       -0.958   
2 -0.683 -0.541               -1.013           1.285        0.281   
3  1.110  0.342               -0.414           2.176       -3.632   
4 -0.004  0.211               -0.534           0.813       -0.241   

   1RM_strength  plank_test  Vertical_jump  Balance  \
0         0.018       0.581          0.603        0   
1         0.060       0.081         -1.068        0   
2        -0.280       0.289          0.407        0   
3         2.313      -0.613          0.073        1   
4        -0.110       0.650         -0.045        0   

   Previous_Physical_Activity_Level  History_Previous_Injury  
0                             0.379                        0  
1                             0.763                        0  
2    

## 4. Fase Online — Inferencia KNN para Nuevo Atleta

### 4.1 Entrenamiento del Motor (Setup Offline Final)

Se entrenan dos clasificadores KNN independientes sobre el espacio vectorial normalizado (`df_inputs`), cada uno apuntando a un target distinto:

- **`knn_restriccion`** → predice `Equipment_Restriction` (`Free_Weight_OK` / `Restricted_Weight`)
- **`knn_riesgo`** → predice `MSI_Overuse` (probabilidad de lesión por sobreuso)

Ambos usan **K=5 vecinos** con **distancia Euclidiana**, apropiada para métricas físicas estáticas normalizadas.

### 4.2 Regla de Oro: `transform`, nunca `fit`

El atleta nuevo se normaliza usando el `scaler` ya ajustado sobre los datos históricos. Reajustar el escalador con datos nuevos desplazaría el espacio vectorial, invalidando las distancias calculadas.

### 4.3 Reporte Dual — Parte 1: Alerta de Riesgo

La inferencia produce dos outputs complementarios que constituyen la primera parte del reporte:

| Output | Fuente | Significado |
|---|---|---|
| Alerta `MSI_Overuse` | Moda de 5 vecinos | Similitud con perfiles históricos de lesión por sobreuso |
| `Equipment_Restriction` | Moda de 5 vecinos | Restricción de equipamiento que se aplicará al catálogo |

La distancia promedio a los vecinos se reporta como **justificación matemática** de la recomendación.

In [15]:
#-------------------------------------------------------------------
# 2. ENTRENAMIENTO DEL MOTOR KNN
# -------------------------------------------------------------------
X_train = df_inputs
y_train_restriccion = df_targets['Equipment_Restriction']
y_train_riesgo      = df_targets['MSI_Overuse']

k_vecinos = 5
knn_restriccion  = KNeighborsClassifier(n_neighbors=k_vecinos, metric='euclidean')
knn_riesgo       = KNeighborsClassifier(n_neighbors=k_vecinos, metric='euclidean')
buscador_vecinos = NearestNeighbors(n_neighbors=k_vecinos, metric='euclidean')

knn_restriccion.fit(X_train, y_train_restriccion)
knn_riesgo.fit(X_train, y_train_riesgo)
buscador_vecinos.fit(X_train)

# -------------------------------------------------------------------
# 3. INGESTA DEL NUEVO ATLETA
# -------------------------------------------------------------------
datos_nuevo_atleta = {
    'mass': 85.0,
    'BMC': 24.0,
    'Body_fat_percentual': 22.0,
    '12-min_running': 2100,
    'flexibility': 15.0,
    '1RM_strength': 90,
    'plank_test': 60,
    'Vertical_jump': 22.0,
    'Balance': 'yes',                       # Binaria: yes/no
    'Previous_Physical_Activity_Level': 42, # Ordinal numérica
    'History_Previous_Injury': 'yes'        # Binaria: yes/no
}

df_nuevo_atleta = pd.DataFrame([datos_nuevo_atleta])

# -------------------------------------------------------------------
# 4. PREPROCESAMIENTO EN CALIENTE
# -------------------------------------------------------------------
# Binarias → 0/1
for col in ['Balance', 'History_Previous_Injury']:
    df_nuevo_atleta[col] = df_nuevo_atleta[col].apply(
        lambda x: 1 if str(x).strip().lower() == 'yes' else 0
    )

# Continuas → transform con el scaler ya ajustado (NUNCA fit)
df_nuevo_atleta[cols_a_escalar] = scaler.transform(df_nuevo_atleta[cols_a_escalar])

# -------------------------------------------------------------------
# 5. INFERENCIA Y REPORTE DUAL — PARTE 1
# -------------------------------------------------------------------
prediccion_riesgo      = knn_riesgo.predict(df_nuevo_atleta)[0]
prediccion_restriccion = knn_restriccion.predict(df_nuevo_atleta)[0]
distancias, indices    = buscador_vecinos.kneighbors(df_nuevo_atleta)

print("=" * 60)
print("  🚨 REPORTE DUAL - FASE 1: EVALUACIÓN DE RIESGO (KNN)")
print("=" * 60)

if str(prediccion_riesgo).strip().lower() == 'yes':
    print("⚠️  ALERTA: Alta similitud con perfiles históricos de MSI_Overuse.")
else:
    print("✅  PERFIL SANO: Baja probabilidad histórica de lesión por sobreuso.")

print(f"🔧  RESTRICCIÓN DE EQUIPAMIENTO: {prediccion_restriccion}")
print("-" * 60)
print(f"Justificación (Distancia Euclidiana promedio a {k_vecinos} vecinos): "
      f"{distancias[0].mean():.2f}")
print("=" * 60)

  🚨 REPORTE DUAL - FASE 1: EVALUACIÓN DE RIESGO (KNN)
✅  PERFIL SANO: Baja probabilidad histórica de lesión por sobreuso.
🔧  RESTRICCIÓN DE EQUIPAMIENTO: Free_Weight_OK
------------------------------------------------------------
Justificación (Distancia Euclidiana promedio a 5 vecinos): 2.37


In [16]:
# Calcular τ como P95 de distancias intra-entrenamiento
distancias_train, _ = buscador_vecinos.kneighbors(X_train)
distancias_promedio_train = distancias_train.mean(axis=1)
tau = np.percentile(distancias_promedio_train, 95)
print(f"Umbral de confianza τ (P95): {tau:.2f}")

Umbral de confianza τ (P95): 0.32


## 5. Fase Offline de Ítems — Vectorización TF-IDF del Catálogo de Ejercicios

### 5.1 Procesamiento del megaGymDataset

Cada ejercicio del catálogo se representa como un único "documento" textual
concatenando sus atributos más relevantes:

```
metadata_combinada = BodyPart + Equipment + Type + Desc
```

Este enfoque permite que el vectorizador capture tanto categorías rígidas
(`BodyPart`, `Equipment`) como contexto semántico libre (`Desc`) en un
único espacio vectorial de términos.

In [24]:
!pip install sentence-transformers

In [17]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "megaGymDataset.csv"

# Load the latest version
df_gym = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "niharika41298/gym-exercise-data",
  file_path,
  # Provide any additional arguments like
  # sql_query or pandas_kwargs. See the
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

print("First 5 records:", df_gym.head())

/tmp/ipykernel_3414/3277148425.py:10: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df_gym = kagglehub.load_dataset(


Using Colab cache for faster access to the 'gym-exercise-data' dataset.
First 5 records:    Unnamed: 0                         Title  \
0           0        Partner plank band row   
1           1  Banded crunch isometric hold   
2           2         FYR Banded Plank Jack   
3           3                 Banded crunch   
4           4                        Crunch   

                                                Desc      Type    BodyPart  \
0  The partner plank band row is an abdominal exe...  Strength  Abdominals   
1  The banded crunch isometric hold is an exercis...  Strength  Abdominals   
2  The banded plank jack is a variation on the pl...  Strength  Abdominals   
3  The banded crunch is an exercise targeting the...  Strength  Abdominals   
4  The crunch is a popular core exercise targetin...  Strength  Abdominals   

  Equipment         Level  Rating RatingDesc  
0     Bands  Intermediate     0.0        NaN  
1     Bands  Intermediate     NaN        NaN  
2     Bands  Inter

In [29]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np
import re

# ===================================================================
# FASE OFFLINE: PROCESAMIENTO Y REPRESENTACIÓN DE ÍTEMS
# ===================================================================

# 1. Limpieza y preparación
df_gym = df_gym.dropna(subset=['Title', 'Desc', 'BodyPart', 'Equipment']).reset_index(drop=True)

df_gym['metadata_combinada'] = (
    (df_gym['BodyPart'] + " ") * 3 +
    (df_gym['Equipment'] + " ") * 3 +
    df_gym['Type'] + " " +
    df_gym['Desc']
)

# 2. VECTORIZACIÓN NEURONAL (Embeddings Densos)
print("Cargando modelo neuronal ligero (MiniLM)...")
semantic_model = SentenceTransformer('all-MiniLM-L6-v2')

print("Generando Embeddings Densos para el catálogo...")
embeddings_matriz = semantic_model.encode(df_gym['metadata_combinada'].tolist(), show_progress_bar=True)

print(f"Catálogo vectorizado: {embeddings_matriz.shape[0]} ejercicios, {embeddings_matriz.shape[1]} dimensiones conceptuales.")

# ===================================================================
# FASE ONLINE: MOTOR DE REGLAS Y SIMILITUD
# ===================================================================

def limpiar_texto_comercial(texto):
    """Elimina sufijos y prefijos comerciales que envenenan la diversidad léxica."""
    t = str(texto).lower().replace('-', ' ')
    basura = [r'gethin variation', r'yates variation', r'variation gethin',
              r'\btbs\b', r'\bfyr\b', r'\bam\b', r'variation']

    for b in basura:
        t = re.sub(b, '', t)
    return " ".join(t.split())

def similitud_jaccard_titulos(titulo1, titulo2):
    """Calcula la intersección matemática de palabras entre dos títulos limpios."""
    set1 = set(limpiar_texto_comercial(titulo1).split())
    set2 = set(limpiar_texto_comercial(titulo2).split())
    if not set1 or not set2:
        return 0
    return len(set1.intersection(set2)) / len(set1.union(set2))

def generar_plan_sustitucion(ejercicio_planeado, restriccion_knn, df_catalogo, matriz_embeddings):
    if ejercicio_planeado not in df_catalogo['Title'].values:
        return pd.DataFrame({"Error": ["El ejercicio base no existe en el catálogo."]})

    idx_original = df_catalogo[df_catalogo['Title'] == ejercicio_planeado].index[0]
    bodypart_original = df_catalogo.loc[idx_original, 'BodyPart']

    # FIX VITAL: Redimensionar el vector de 1D a 2D para que scikit-learn lo acepte
    vector_original = matriz_embeddings[idx_original].reshape(1, -1)

    # Similitud del Coseno sobre Embeddings
    similitudes = cosine_similarity(vector_original, matriz_embeddings).flatten()

    df_temp = df_catalogo.copy()
    df_temp['Cosine_Similarity'] = similitudes

    # Filtros Duros (Músculo exacto y exclusión del ejercicio original)
    df_temp = df_temp[df_temp['Title'] != ejercicio_planeado]
    df_temp = df_temp[df_temp['BodyPart'] == bodypart_original]

    # Filtro Kinesiológico de Equipamiento
    if restriccion_knn == 'Restricted_Weight':
        equipos_seguros = ['Machine', 'Cable', 'Bands']
        df_temp = df_temp[df_temp['Equipment'].isin(equipos_seguros)]

    df_temp = df_temp.sort_values(by='Cosine_Similarity', ascending=False)

    top_3_diversos = []

    for _, row in df_temp.iterrows():
        # Los embeddings densos suelen dar cosenos más altos. 0.98 indica copia exacta.
        if row['Cosine_Similarity'] >= 0.98:
            continue

        titulo_candidato = row['Title']
        es_diverso = True

        # 1. Comparar candidato contra el EJERCICIO BASE (Anti-Clones comerciales)
        if similitud_jaccard_titulos(titulo_candidato, ejercicio_planeado) > 0.65:
            continue

        # 2. Comparar candidato contra los que ya están en el Top 3 (Diversidad interna)
        for rec in top_3_diversos:
            if similitud_jaccard_titulos(titulo_candidato, rec['Title']) > 0.40:
                es_diverso = False
                break

        if es_diverso:
            top_3_diversos.append(row)

        if len(top_3_diversos) == 3:
            break

    df_final = pd.DataFrame(top_3_diversos)

    # Manejo de catálogos exhaustos (Dead Ends)
    if df_final.empty:
        return pd.DataFrame({
            "Title": ["Catálogo exhausto para este BodyPart bajo restricción"],
            "BodyPart": [bodypart_original],
            "Equipment": ["N/A"],
            "Cosine_Similarity": [0.0]
        })

    return df_final[['Title', 'BodyPart', 'Equipment', 'Cosine_Similarity']]

# ===================================================================
# EJECUCIÓN FINAL (Prueba en Caliente)
# ===================================================================

ejercicio_input = "Barbell roll-out"
restriccion_calculada = "Restricted_Weight"

print("\n" + "="*60)
print(" 🔄 REPORTE DUAL - FASE 2: PLAN DE SUSTITUCIÓN")
print("="*60)
print(f"Ejercicio Planeado: [{ejercicio_input}]")
print(f"Restricción de Sistema: [{restriccion_calculada}]\n")

recomendaciones_finales = generar_plan_sustitucion(
    ejercicio_planeado=ejercicio_input,
    restriccion_knn=restriccion_calculada,
    df_catalogo=df_gym,
    matriz_embeddings=embeddings_matriz
)

print(recomendaciones_finales)
print("="*60)

Cargando modelo neuronal ligero (MiniLM)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generando Embeddings Densos para el catálogo...


Batches:   0%|          | 0/43 [00:00<?, ?it/s]

Catálogo vectorizado: 1359 ejercicios, 384 dimensiones conceptuales.

 🔄 REPORTE DUAL - FASE 2: PLAN DE SUSTITUCIÓN
Ejercicio Planeado: [Barbell roll-out]
Restricción de Sistema: [Restricted_Weight]

                   Title    BodyPart Equipment  Cosine_Similarity
85     Ab Crunch Machine  Abdominals   Machine           0.635369
78  30 Cable Rope Crunch  Abdominals     Cable           0.616518
4                 Crunch  Abdominals     Bands           0.615602


## 6. Capa de Presentación — Interfaz Web con Gradio

### 6.1 Arquitectura de la Interfaz

Gradio envuelve todo el pipeline como una función Python única
(`interfaz_entrenador`) y genera automáticamente una página web interactiva
accesible desde un enlace público temporal (`*.gradio.live`), sin necesidad
de configurar servidores ni frontend independiente.

### 6.2 Inputs del Entrenador

| Input | Tipo de control | Rango / Opciones | Fuente del rango |
|---|---|---|---|
| Peso corporal | Slider | 40–120 kg | Dataset militar |
| % Grasa corporal | Slider | 3–40% | Dataset militar |
| Fuerza 1RM | Slider | 20–200 kg | Dataset militar |
| Flexibilidad | Slider | 1–50 cm | Dataset militar |
| Nivel actividad previa | Slider | 26–65 | Dataset militar (ordinal numérico) |
| Déficit de balance | Radio | Sí / No | Variable clínica binaria |
| Lesión previa | Radio | Sí / No | Variable clínica binaria |
| Ejercicio planeado | Dropdown | 1,359 opciones | `megaGymDataset` |

> **Simplificación de UI documentada:** Las variables `BMC`, `12-min_running`,
> `plank_test` y `Vertical_jump` se fijan en sus valores promedio poblacional
> (extraídos de `df_final.describe()`) para reducir la carga cognitiva del
> entrenador en la interfaz. Esta decisión de diseño es válida para un prototipo
> y debe mencionarse explícitamente en la sustentación.

### 6.3 Outputs del Reporte Dual

| Output | Módulo origen | Contenido |
|---|---|---|
| Alerta de Riesgo | KNN — espacio vectorial de usuarios | Clasificación `MSI_Overuse` + etiqueta `Equipment_Restriction` |
| Top 3 Sustitutos | TF-IDF + Similitud Coseno — espacio vectorial de ítems | Ejercicios seguros ordenados por similitud biomecánica |

In [19]:
!pip install gradio

In [34]:
import gradio as gr
import pandas as pd
import numpy as np
import re
from sklearn.metrics.pairwise import cosine_similarity

# --- NOTA PARA EJECUCIÓN ---
# Este código asume que ya tienes cargados en memoria:
# 1. semantic_model (SentenceTransformer)
# 2. embeddings_matriz (Numpy array de descriptores del catálogo)
# 3. knn_riesgo y knn_restriccion (Modelos entrenados)
# 4. scaler (StandardScaler ajustado)
# 5. df_gym (DataFrame del catálogo)
# ---------------------------

def limpiar_texto_comercial(texto):
    t = str(texto).lower().replace('-', ' ')
    basura = [r'gethin variation', r'yates variation', r'variation gethin',
              r'\btbs\b', r'\bfyr\b', r'\bam\b', r'variation']
    for b in basura:
        t = re.sub(b, '', t)
    return " ".join(t.split())

def similitud_jaccard_titulos(titulo1, titulo2):
    set1 = set(limpiar_texto_comercial(titulo1).split())
    set2 = set(limpiar_texto_comercial(titulo2).split())
    if not set1 or not set2: return 0
    return len(set1.intersection(set2)) / len(set1.union(set2))

def generar_plan_sustitucion(ejercicio_planeado, restriccion_knn, df_catalogo, matriz_embeddings):
    if ejercicio_planeado not in df_catalogo['Title'].values:
        return pd.DataFrame({"Error": ["El ejercicio base no existe en el catálogo."]})

    idx_original = df_catalogo[df_catalogo['Title'] == ejercicio_planeado].index[0]
    bodypart_original = df_catalogo.loc[idx_original, 'BodyPart']
    vector_original = matriz_embeddings[idx_original].reshape(1, -1)

    # Similitud sobre Embeddings Neuronales
    similitudes = cosine_similarity(vector_original, matriz_embeddings).flatten()

    df_temp = df_catalogo.copy()
    df_temp['Cosine_Similarity'] = similitudes
    df_temp = df_temp[df_temp['Title'] != ejercicio_planeado]
    df_temp = df_temp[df_temp['BodyPart'] == bodypart_original]

    if restriccion_knn == 'Restricted_Weight':
        equipos_seguros = ['Machine', 'Cable', 'Bands']
        df_temp = df_temp[df_temp['Equipment'].isin(equipos_seguros)]

    df_temp = df_temp.sort_values(by='Cosine_Similarity', ascending=False)

    top_3_diversos = []
    for _, row in df_temp.iterrows():
        if row['Cosine_Similarity'] >= 0.98: continue

        titulo_candidato = row['Title']
        es_diverso = True

        # Jaccard contra el original para evitar clones
        if similitud_jaccard_titulos(titulo_candidato, ejercicio_planeado) > 0.65:
            continue

        # Jaccard interno para variedad en el Top 3
        for rec in top_3_diversos:
            if similitud_jaccard_titulos(titulo_candidato, rec['Title']) > 0.40:
                es_diverso = False
                break

        if es_diverso:
            top_3_diversos.append(row)

        if len(top_3_diversos) == 3: break

    df_final = pd.DataFrame(top_3_diversos)
    if df_final.empty:
        return pd.DataFrame({"Title": ["Catálogo exhausto para este perfil"], "BodyPart": [bodypart_original], "Equipment": ["N/A"], "Cosine_Similarity": [0.0]})

    return df_final[['Title', 'BodyPart', 'Equipment', 'Cosine_Similarity']]

def interfaz_entrenador(mass, body_fat, rm_strength, flexibility,
                        prev_activity_level, balance_deficit,
                        prev_injury, ejercicio_planeado):

    datos_nuevo_atleta = {
        'mass': float(mass),
        'BMC': 22.0,
        'Body_fat_percentual': float(body_fat),
        '12-min_running': 2581,
        'flexibility': float(flexibility),
        '1RM_strength': int(rm_strength),
        'plank_test': 184,
        'Vertical_jump': 33.9,
        'Balance': 1 if balance_deficit == "Sí" else 0,
        'Previous_Physical_Activity_Level': int(prev_activity_level),
        'History_Previous_Injury': 1 if prev_injury == "Sí" else 0
    }

    df_nuevo = pd.DataFrame([datos_nuevo_atleta])
    # Escalar métricas continuas (cols_a_escalar debe estar definido previamente)
    df_nuevo[cols_a_escalar] = scaler.transform(df_nuevo[cols_a_escalar])

    prediccion_riesgo      = knn_riesgo.predict(df_nuevo)[0]
    prediccion_restriccion = knn_restriccion.predict(df_nuevo)[0]

    if str(prediccion_riesgo).strip().lower() == 'yes':
        alerta = f"⚠️ ALERTA: Alta similitud con perfiles de lesiones por sobreuso.\n🔧 Restricción: {prediccion_restriccion}"
    else:
        alerta = f"✅ PERFIL SANO: Baja probabilidad de lesión.\n🔧 Restricción: {prediccion_restriccion}"

    try:
        df_recomendaciones = generar_plan_sustitucion(
            ejercicio_planeado=ejercicio_planeado,
            restriccion_knn=prediccion_restriccion,
            df_catalogo=df_gym,
            matriz_embeddings=embeddings_matriz
        )
    except Exception as e:
        df_recomendaciones = pd.DataFrame({"Error": [str(e)]})

    return alerta, df_recomendaciones

# --- CONFIGURACIÓN DE LA INTERFAZ CON EMOJIS ---

# Diccionarios de mapeo visual
EMOJIS_BODYPART = {
    'Abdominals': '🍫', 'Biceps': '💪', 'Triceps': '💪', 'Chest': '👕',
    'Lats': '🔙', 'Lower Back': '🔙', 'Middle Back': '🔙', 'Traps': '⛰️',
    'Shoulders': '🥥', 'Quadriceps': '🦵', 'Hamstrings': '🦵',
    'Calves': '🦿', 'Glutes': '🍑', 'Forearms': '🦾', 'Neck': '🦒',
    'Abductors': '🤸', 'Adductors': '🤸'
}

EMOJIS_EQUIPMENT = {
    'Barbell': '🏋️', 'Dumbbell': '🔩', 'Machine': '⚙️', 'Cable': '⛓️',
    'Bands': '〰️', 'Body Only': '🏃', 'Kettlebells': '💣',
    'Exercise Ball': '⚽', 'E-Z Curl Bar': '〰️', 'Medicine Ball': '🏀',
    'Other': '🔹'
}

# Crear lista de tuplas (Texto_Visual_UI, Valor_Interno_Backend)
lista_ejercicios_ui = []
df_unicos = df_gym.drop_duplicates(subset=['Title']).sort_values(by='Title')

for _, row in df_unicos.iterrows():
    titulo = row['Title']
    bp = row['BodyPart']
    eq = row['Equipment']

    emoji_bp = EMOJIS_BODYPART.get(bp, '👤')
    emoji_eq = EMOJIS_EQUIPMENT.get(eq, '🔧')

    # Formato visual en el dropdown: 🏋️ 👕 Barbell Bench Press [Chest | Barbell]
    etiqueta_visual = f"{emoji_eq} {emoji_bp} {titulo} [{bp} | {eq}]"

    # Gradio asigna el primer valor a la vista y el segundo al backend
    lista_ejercicios_ui.append((etiqueta_visual, titulo))


demo = gr.Interface(
    fn=interfaz_entrenador,
    title="⚕️ SR Preventivo Clínico-Deportivo (Demo)",
    description="Caso predeterminado: Usuario pesado, baja movilidad, intentando Press de Banca con Barra.",
    inputs=[
        gr.Slider(40, 120, value=105,  label="Peso Corporal (kg)"),
        gr.Slider(3,  40,  value=28,   label="% Grasa Corporal"),
        gr.Slider(20, 200, value=140,  label="Fuerza Base — 1RM (kg)"),
        gr.Slider(1,  50,  value=5,    label="Flexibilidad (cm)"),
        gr.Slider(26, 65,  value=30,   label="Nivel de Actividad Física Previa"),
        gr.Radio(["Sí", "No"], value="No", label="¿Presenta Déficit de Balance?"),
        gr.Radio(["Sí", "No"], value="No", label="¿Historial de Lesión Previa?"),

        # Pasamos la lista de tuplas. Gradio entiende automáticamente el formato.
        gr.Dropdown(choices=lista_ejercicios_ui,
                    value="Barbell Bench Press - Medium Grip",
                    label="Ejercicio Planeado",
                    allow_custom_value=False)
    ],
    outputs=[
        gr.Textbox(label="1. Alerta de Riesgo (KNN — Perfil de Usuario)"),
        gr.Dataframe(label="2. Top 3 Sustitutos Seguros (Embeddings Semánticos)")
    ],
    theme="huggingface"
)

demo.launch(share=True)

/usr/local/lib/python3.12/dist-packages/gradio/blocks.py:1143: UserWarning: Cannot load huggingface. Caught Exception: Client error '404 Not Found' for url 'https://huggingface.co/api/spaces/huggingface' (Request ID: Root=1-69f185c9-560ae7543dfeda1b3d9f6ae3;8f7c325f-a542-488c-9267-41b152f5dcb7)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404

Sorry, we can't find the page you are looking for.
  warnings.warn(f"Cannot load {theme}. Caught Exception: {str(e)}")


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0af731e85c911c90fd.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [31]:
import pandas as pd
import json

# 1. Tomar muestra aleatoria del catálogo (Ajusta n según la cantidad que quieras auditar)
# Usamos random_state para reproducibilidad. Si quitas random_state, será distinto cada vez.
muestra_aleatoria = df_gym.sample(n=15, random_state=42)

resultados_aleatorios = []

print(f"Iniciando estrés semántico con {len(muestra_aleatoria)} ejercicios aleatorios...\n")

for _, row in muestra_aleatoria.iterrows():
    ejercicio_base = row['Title']
    bodypart_base = row['BodyPart']
    equipo_base = row['Equipment']

    caso = {
        "1_Ejercicio_Base": f"{ejercicio_base} | BodyPart: {bodypart_base} | Eq: {equipo_base}",
        "2_Escenario_SANO (Free_Weight_OK)": [],
        "3_Escenario_RIESGO (Restricted_Weight)": []
    }

    # -- PRUEBA 1: Sin restricción (Catálogo completo) --
    try:
        df_sano = generar_plan_sustitucion(ejercicio_base, "Free_Weight_OK", df_gym, tfidf_matrix)
        if "Error" not in df_sano.columns and "Alerta" not in df_sano.columns:
            caso["2_Escenario_SANO (Free_Weight_OK)"] = df_sano.apply(
                lambda x: f"{x['Title']} ({x['Equipment']}) - Sim: {x['Cosine_Similarity']:.3f}", axis=1
            ).tolist()
        else:
            caso["2_Escenario_SANO (Free_Weight_OK)"] = df_sano.to_dict(orient="records")
    except Exception as e:
        caso["2_Escenario_SANO (Free_Weight_OK)"] = [f"ERROR: {str(e)}"]

    # -- PRUEBA 2: Con restricción (Filtro Kinesiológico activado) --
    try:
        df_riesgo = generar_plan_sustitucion(ejercicio_base, "Restricted_Weight", df_gym, tfidf_matrix)
        if "Error" not in df_riesgo.columns and "Alerta" not in df_riesgo.columns:
            caso["3_Escenario_RIESGO (Restricted_Weight)"] = df_riesgo.apply(
                lambda x: f"{x['Title']} ({x['Equipment']}) - Sim: {x['Cosine_Similarity']:.3f}", axis=1
            ).tolist()
        else:
             caso["3_Escenario_RIESGO (Restricted_Weight)"] = df_riesgo.to_dict(orient="records")
    except Exception as e:
        caso["3_Escenario_RIESGO (Restricted_Weight)"] = [f"ERROR: {str(e)}"]

    resultados_aleatorios.append(caso)

# 2. Imprimir salida procesable
salida_json = json.dumps(resultados_aleatorios, indent=2, ensure_ascii=False)
print("=== COPIA ESTE BLOQUE JSON ===")
print(salida_json)
print("==============================")

Iniciando estrés semántico con 15 ejercicios aleatorios...

=== COPIA ESTE BLOQUE JSON ===
[
  {
    "1_Ejercicio_Base": "Single-arm bent-over rear delt fly | BodyPart: Abdominals | Eq: Other",
    "2_Escenario_SANO (Free_Weight_OK)": [
      "Decline crunch- (Body Only) - Sim: 0.324",
      "Decline sit-up twist (Body Only) - Sim: 0.267",
      "Bent-knee reverse crunch (Body Only) - Sim: 0.252"
    ],
    "3_Escenario_RIESGO (Restricted_Weight)": [
      "Cable reverse crunch (Cable) - Sim: 0.181",
      "Standing cable low-to-high twist (Cable) - Sim: 0.181",
      "Standing Cable Wood Chop (Cable) - Sim: 0.160"
    ]
  },
  {
    "1_Ejercicio_Base": "Band upright row | BodyPart: Shoulders | Eq: Bands",
    "2_Escenario_SANO (Free_Weight_OK)": [
      "Single-arm dumbbell upright row (Dumbbell) - Sim: 0.673",
      "Barbell upright row (Barbell) - Sim: 0.553",
      "Band lateral raise (Bands) - Sim: 0.414"
    ],
    "3_Escenario_RIESGO (Restricted_Weight)": [
      "Band lateral r